# Training the Genre and Language LoRA Modules

Backbone: **`facebook/xglm-564M`** — a 564M-parameter decoder-only multilingual LM (standard
multi-head attention, no grouped-query attention)


**Module = LoRA adapter.** Each factor (genre 𝑔, language 𝑙) gets its own low-rank adapter on top
of the same frozen backbone, trained completely independently on the data prepared in
`01_data_extraction.ipynb`:

- **genre module** ← `data/genre_module_dgt_non_fi/` (OPUS DGT, all languages except Finnish)
- **language module** ← `data/language_module_books_fi/` (OPUS Books, Finnish only)

This notebook only trains thetwo modules; composing them and evaluating on the target domain is in a separate
notebook.


## 0. Setup

In [ ]:
import gc
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

MODEL_NAME = "facebook/xglm-564M"
DATA_DIR = Path("data")            # written by 01_data_extraction.ipynb
MODELS_DIR = Path("models")        # where each trained LoRA adapter gets saved
MODELS_DIR.mkdir(exist_ok=True)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Config


In [ ]:
MAX_LENGTH = 128            # covers ~90th percentile of sentence lengths in both corpora

# LoRA hyperparameters, identical for BOTH modules, since later are combines in
# weight space (ΔW_genre + ΔW_language); that only makes sense if both deltas live in a
# comparable low-rank subspace
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]  # xglm's attention proj names

LEARNING_RATE = 2e-4
BATCH_SIZE = 4
MAX_STEPS = 600           
                              
WARMUP_STEPS = 30
LOGGING_STEPS = 50
EVAL_STEPS = 150
EVAL_SUBSET_SIZE = 200       

## 2. Tokenizer (shared, loaded once)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # XGLM has no dedicated pad token

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH)

## 3. Helpers

In [ ]:
def load_tokenized_split(path, max_examples=None):
    ds = load_dataset("json", data_files=str(path), split="train")
    if max_examples is not None and len(ds) > max_examples:
        ds = ds.shuffle(seed=SEED).select(range(max_examples))
    return ds.map(tokenize, remove_columns=ds.column_names)


def fresh_lora_model():
    """A new frozen backbone + a freshly-initialized LoRA adapter, so every module is
    trained from scratch with zero state shared with any previously trained module."""
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)  # freezes the backbone, adds trainable LoRA A/B matrices
    model.to(DEVICE)
    return model


def free(*objs):
    """Cleanup between the two module-training runs."""
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()


def train_module(name, train_path, dev_path, output_dir):
    print(f"\n=== training {name} module ===")
    train_ds = load_tokenized_split(train_path)
    dev_ds = load_tokenized_split(dev_path, max_examples=EVAL_SUBSET_SIZE)
    print(f"train examples: {len(train_ds):,} | eval examples: {len(dev_ds):,}")

    model = fresh_lora_model()
    model.print_trainable_parameters()  # sanity check: should be a small fraction of all params

    # mlm=False -> causal LM collation: labels are a (shifted) copy of input_ids, pad positions masked
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=f"runs/{name}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="no",   # adapter is saved manually below; no need for HF checkpoints
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        data_collator=collator,
    )

    # LoRA's B matrix is zero-initialized, so the model at step 0 is numerically identical
    # to the plain backbone — this eval *is* the pre-adaptation baseline, no extra model load needed.
    baseline = trainer.evaluate()
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    final = trainer.evaluate()

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)   # only the small LoRA adapter is written, not the backbone
    tokenizer.save_pretrained(output_dir)
    print(f"saved adapter to {output_dir} ({elapsed/60:.1f} min)")

    result = {
        "module": name,
        "baseline_loss": baseline["eval_loss"],
        "final_loss": final["eval_loss"],
        "minutes": elapsed / 60,
    }

    free(model, trainer)
    return result

## 4. Train the genre module

`g` = legal/administrative register, learned from OPUS DGT in every available language **except**
Finnish (𝐿∖𝑙).


In [ ]:
# g = legal/administrative register, learned from every DGT language EXCEPT Finnish (L∖l)
genre_result = train_module(
    name="genre",
    train_path=DATA_DIR / "genre_module_dgt_non_fi" / "train.jsonl",
    dev_path=DATA_DIR / "genre_module_dgt_non_fi" / "dev.jsonl",
    output_dir=MODELS_DIR / "genre_module_dgt",
)
genre_result


=== training genre module ===



Map:   0%|          | 0/445500 [00:00<?, ? examples/s]


Map:   0%|          | 759/445500 [00:00<00:58, 7552.30 examples/s]


Map:   0%|          | 1939/445500 [00:00<00:57, 7759.22 examples/s]


Map:   1%|          | 2810/445500 [00:00<00:54, 8124.70 examples/s]

...

trainable params: 1,572,864 || all params: 566,036,480 || trainable%: 0.2779


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Model Preparation Time
150,7.036700,6.927866,0.004600
300,6.664400,6.759491,0.004600
450,6.916900,6.679986,0.004600
600,6.736000,6.656613,0.004600


saved adapter to models/genre_module_dgt (13.1 min)


{'module': 'genre',
 'baseline_loss': 7.896988391876221,
 'final_loss': 6.656613349914551,
 'minutes': 13.142190432548523}

## 5. Train the language module

`l` = Finnish, learned from OPUS Books (¬𝑔, the literary genre) — the small, ~64k-token corpus
flagged in the previous notebook.


In [ ]:
# l = Finnish, learned from OPUS Books (¬g, the literary genre) 
language_result = train_module(
    name="language",
    train_path=DATA_DIR / "language_module_books_fi" / "train.jsonl",
    dev_path=DATA_DIR / "language_module_books_fi" / "dev.jsonl",
    output_dir=MODELS_DIR / "language_module_books_fi",
)
language_result


=== training language module ===



Map:   0%|          | 0/4285 [00:00<?, ? examples/s]


Map:   1%|          | 47/4285 [00:00<00:09, 438.57 examples/s]


Map:  23%|██▎       | 1000/4285 [00:00<00:00, 5080.52 examples/s]


Map:  49%|████▊     | 2083/4285 [00:00<00:00, 7516.24 examples/s]

...

train examples: 4,285 | eval examples: 200


trainable params: 1,572,864 || all params: 566,036,480 || trainable%: 0.2779


Step,Training Loss,Validation Loss,Model Preparation Time
150,4.274700,4.062219,0.004500
300,4.103400,3.893948,0.004500
450,4.006600,3.817938,0.004500
600,4.032100,3.803301,0.004500


saved adapter to models/language_module_books_fi (10.1 min)


{'module': 'language',
 'baseline_loss': 5.626596450805664,
 'final_loss': 3.8033013343811035,
 'minutes': 10.104913079738617}

## 6. Summary

Both baseline numbers are the plain backbone's loss on that module's own dev set


In [7]:
import pandas as pd

summary = pd.DataFrame([genre_result, language_result])
# perplexity = exp(mean NLL); using e explicitly here since the loss is natural-log cross-entropy
summary["ppl_baseline"] = summary["baseline_loss"].apply(lambda x: round(pow(2.718281828, x), 2))
summary["ppl_final"] = summary["final_loss"].apply(lambda x: round(pow(2.718281828, x), 2))
summary

,module,baseline_loss,final_loss,minutes,ppl_baseline,ppl_final
0,genre,7.896988,6.656613,13.142190,2689.17,777.91
1,language,5.626596,3.803301,10.104913,277.72,44.85


In [8]:
# confirms only the small LoRA deltas were saved (a few MB), not full backbone copies
for d in [MODELS_DIR / "genre_module_dgt", MODELS_DIR / "language_module_books_fi"]:
    size_mb = sum(f.stat().st_size for f in d.glob("*.safetensors")) / 1e6
    print(f"{d}: {size_mb:.1f} MB (adapter weights only)")

models/genre_module_dgt: 6.3 MB (adapter weights only)
models/language_module_books_fi: 6.3 MB (adapter weights only)
